# Phishing Website Detection — Step 1: Getting & Exploring the Dataset

**Capstone roadmap (where we are):**

1. **Step 1 (this notebook): Dataset acquisition & exploration**
2. Feature understanding + baseline model
3. Model evaluation & improvement
4. Wrap model in an API
5. Build the browser extension that calls the API

### What are we doing in this notebook?

Before training any model, we need to understand our data. This notebook will:

- Load a well-known phishing detection dataset (from the UCI Machine Learning Repository)
- Look at what the data actually contains — how many samples, how many features, what the features mean
- Check the data quality (missing values, class balance)
- Get a visual feel for the data before we start modeling

**Why this matters:** a model is only as good as the data you feed it. Skipping this step is the #1 reason beginner ML projects produce misleading results (e.g. a model that looks 95% accurate but is actually useless).

## 1. About the dataset

We're using the **UCI "Phishing Websites" dataset** (Mohammad & McCluskey, 2012):

- 11,055 websites total
- 30 pre-extracted features per website (so we don't have to write our own feature-extraction code yet — that comes later when we build the live API)
- A label column telling us whether each site is phishing or legitimate

The 30 features fall into 4 groups — good to know conceptually before you see the column names:

| Group | Examples | Idea |
|---|---|---|
| **Address bar-based** | URL length, use of IP address instead of domain, `@` symbol in URL, use of URL-shortening services | Phishing URLs often look "off" just by structure |
| **Abnormal-based** | Mismatched URL/domain, abnormal request URLs | Signs the site is impersonating something |
| **HTML/JavaScript-based** | Invisible iframes, right-click disabled, pop-up windows | Techniques phishing pages use to hide or manipulate |
| **Domain-based** | Domain age, DNS record existence, website traffic rank | Legitimate sites tend to be older and more established |

Each feature is encoded numerically (mostly as -1, 0, or 1) rather than as raw text — this is already done for us in this dataset.

## 2. Install & import libraries

Run the cell below once. `ucimlrepo` is the official helper library for pulling datasets directly from the UCI repository — no manual downloading needed.

In [ ]:
# Run this once. If you're using a fresh environment (e.g. Google Colab), keep this cell.
!pip install ucimlrepo pandas matplotlib seaborn scikit-learn --quiet

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

# Makes plots a bit nicer by default
sns.set_theme(style="whitegrid")

## 3. Load the dataset

`fetch_ucirepo(id=327)` pulls the exact dataset described above directly from UCI's servers.

In [ ]:
# id=327 is the UCI catalog ID for the "Phishing Websites" dataset
phishing_websites = fetch_ucirepo(id=327)

# X = the 30 feature columns, y = the label column (whether it's phishing or not)
X = phishing_websites.data.features
y = phishing_websites.data.targets

print("Features shape:", X.shape)
print("Target shape:", y.shape)

### Important: let's not assume what the label values mean — let's check

It's tempting to just trust documentation you read online for what `-1` and `1` mean in the target column. Good practice: always confirm it directly from the dataset's own metadata before you build anything on top of an assumption.

In [ ]:
# This prints UCI's own description of the dataset and each variable
# Read through the target ('Result') description carefully to confirm what each label value means
print(phishing_websites.metadata.additional_info.summary)
print("\n--- Variable info ---")
phishing_websites.variables

**Your task:** note down here (edit this cell) what you find — which value means phishing and which means legitimate. We'll need this for Step 2.

`Result = ___`  →  phishing

`Result = ___`  →  legitimate

## 4. First look at the data

In [ ]:
# Combine features and target into one DataFrame so it's easier to explore together
df = X.copy()
df['Result'] = y

df.head()

In [ ]:
# Column names and data types — good to skim through once
df.info()

In [ ]:
# Any missing values? A clean dataset like this usually has none, but always check.
df.isnull().sum().sum()  # total count of missing cells across the whole DataFrame

## 5. Class balance — how many phishing vs. legitimate sites?

This matters a lot. If, say, 95% of the data is legitimate sites, a lazy model could just always predict "legitimate" and still be 95% accurate — while being completely useless. We need to know this *before* we train anything, so later we pick the right evaluation metrics (accuracy alone won't tell the full story).

In [ ]:
df['Result'].value_counts()

In [ ]:
sns.countplot(data=df, x='Result')
plt.title('Class balance: phishing vs. legitimate')
plt.xlabel('Result')
plt.ylabel('Number of websites')
plt.show()

## 6. Which features seem most related to the label?

Since all features here are already numeric, we can compute a correlation of each feature with `Result` as a quick, rough signal of which features might matter most. (Correlation isn't the full picture for a model like Random Forest, but it's a good sanity-check exercise at this stage.)

In [ ]:
correlations = df.corr(numeric_only=True)['Result'].drop('Result').sort_values(key=abs, ascending=False)
correlations.head(10)

In [ ]:
plt.figure(figsize=(8, 6))
correlations.head(10).plot(kind='barh')
plt.title('Top 10 features most correlated with Result')
plt.xlabel('Correlation with Result')
plt.gca().invert_yaxis()
plt.show()

## 7. Save a local copy (optional but recommended)

So we don't have to re-fetch from UCI every time, let's save a CSV copy locally. We'll load this file directly in Step 2.

In [ ]:
df.to_csv('phishing_data.csv', index=False)
print("Saved phishing_data.csv —", df.shape[0], "rows,", df.shape[1], "columns")

## Summary — what we now know

- [ ] How many samples and features the dataset has
- [ ] What the 4 feature groups roughly represent
- [ ] What `Result = -1` and `Result = 1` actually mean (fill this in from Section 3!)
- [ ] Whether the classes are balanced or imbalanced
- [ ] Which features look most correlated with the label

**Next up (Step 2):** we'll split this data into training/test sets, train our first baseline model (Logistic Regression), and learn *why* we evaluate it with more than just accuracy.

Before moving on: run through this whole notebook yourself, fill in the blank in Section 3, and make sure you can explain in your own words what class balance means and why it matters. That understanding is what Step 2 builds on.